# Neural PK-PD Modeling: Data Exploration

**Date**: February 17, 2026  
**Purpose**: Exploratory Data Analysis (EDA) of 5 integrated data sources  
**Status**: Phase 1 - Data Loading & Inspection

---

## Dataset Overview

This notebook explores the following data sources:

1. **PubChem** - Bioassay screening (hERG, CYP3A4)
2. **ChEMBL** - Target binding affinity data
3. **TDC ADMET** - Drug property benchmarks
4. **ToxCast** - Toxicity screening results
5. **PK-DB** - Pharmacokinetic studies & time-courses

**Total**: ~500,000 records across molecular, safety, and PK domains

In [ ]:
# =============================================================================
# CELL: Execution Timestamp Logger (with start + end tracking)
# PURPOSE: Track execution order, timestamps, and duration for every cell
# OUTPUTS: Logs each cell's start/end time; summary cell shows full timeline
# =============================================================================

from datetime import datetime as _dt_cls

CELL_EXEC_LOG: list = []   # [{section, start, end, duration_s}, ...]


def log_cell_start(section: str) -> None:
    """Record cell execution start time."""
    now = _dt_cls.now()
    CELL_EXEC_LOG.append({"section": section, "start": now, "end": None, "duration_s": None})
    print(f"\u23f1  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Starting: {section}")


def log_cell_end() -> None:
    """Record cell execution end time for the most recent entry."""
    now = _dt_cls.now()
    if CELL_EXEC_LOG and CELL_EXEC_LOG[-1]["end"] is None:
        entry = CELL_EXEC_LOG[-1]
        entry["end"] = now
        entry["duration_s"] = round((now - entry["start"]).total_seconds(), 2)
        print(f"\u23f1  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Completed in {entry['duration_s']:.1f}s")


print("\u2705 Timestamp logger ready  \u2192  log_cell_start() / log_cell_end() active")
print("   Run the summary cell at the bottom to view the full timeline.")


In [ ]:

# =============================================================================
# CELL: Library Imports & Configuration
# PURPOSE: Import essential libraries and configure environment settings
# OUTPUTS: Confirmation messages showing library versions
# =============================================================================
log_cell_start("Cell 01 - Library Imports")

import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Data directory
CODING_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'data').exists()),
    Path.cwd(),
)
DATA_DIR = CODING_ROOT / 'data/raw'

print("✅ Libraries loaded successfully")
print(f"📁 Data directory: {DATA_DIR}")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")

log_cell_end()


### 📦 Cell: Library Imports & Configuration

Import essential Python libraries for data analysis and visualization. Configure plotting styles and suppress warnings for cleaner output.

In [ ]:

# =============================================================================
# CELL: Dataset Loading
# PURPOSE: Load all 5 data sources (PubChem, ChEMBL, TDC, ToxCast, PK-DB)
# OUTPUTS: Record counts for each dataset and total summary
# =============================================================================
log_cell_start("Cell 02 - Dataset Loading")

print("Loading datasets...\n")

# 1. PubChem - hERG toxicity assay
pubchem_herg = pd.read_csv(DATA_DIR / 'pubchem/assay_herg_qhts_aid588834.csv')
print(f"✅ PubChem hERG:     {len(pubchem_herg):>8,} assay results")

# 2. PubChem - CYP3A4 metabolism assay
pubchem_cyp = pd.read_csv(DATA_DIR / 'pubchem/assay_cyp3a4_inhibition_aid54772.csv')
print(f"✅ PubChem CYP3A4:   {len(pubchem_cyp):>8,} assay results")

# 3. ChEMBL - Binding affinity data
chembl = pd.read_csv(DATA_DIR / 'chembl/chembl_binding_affinity.csv')
print(f"✅ ChEMBL:           {len(chembl):>8,} binding records")

# 4. TDC ADMET - Drug properties
tdc_herg = pd.read_csv(DATA_DIR / 'tdc/tdc_herg.csv')
tdc_caco2 = pd.read_csv(DATA_DIR / 'tdc/tdc_caco2_wang.csv')
tdc_clearance = pd.read_csv(DATA_DIR / 'tdc/tdc_clearance_hepatocyte_az.csv')
print(f"✅ TDC hERG:         {len(tdc_herg):>8,} compounds")
print(f"✅ TDC Caco-2:       {len(tdc_caco2):>8,} compounds")
print(f"✅ TDC Clearance:    {len(tdc_clearance):>8,} compounds")

# 5. ToxCast - Toxicity screening
toxcast = pd.read_csv(DATA_DIR / 'toxcast/toxcast_representative.csv')
print(f"✅ ToxCast:          {len(toxcast):>8,} toxicity results")

# 6. PK-DB - Pharmacokinetic studies
with open(DATA_DIR / 'pkdb/pkdb_studies_complete.json') as f:
    pkdb = json.load(f)
print(f"✅ PK-DB:            {len(pkdb['studies']):>8,} PK studies")

print(f"\n{'='*50}")
total_records = len(pubchem_herg) + len(pubchem_cyp) + len(chembl) + len(tdc_herg) + len(toxcast)
print(f"TOTAL RECORDS:      {total_records:>8,}")
print(f"{'='*50}")

log_cell_end()


---

## 📋 Section 1: Dataset Inspection

Let's inspect the ChEMBL binding affinity and ToxCast toxicity data to understand their schema and identify any data quality issues.

**Objective**: Examine the structure, data types, and quality of each dataset

In [ ]:

# =============================================================================
# CELL: ChEMBL Data Inspection
# PURPOSE: Examine structure, columns, and statistics of binding affinity data
# OUTPUTS: Data shape, column list, sample records, missing values, summary stats
# =============================================================================
log_cell_start("Cell 03 - ChEMBL Inspection")

print("=" * 70)
print("ChEMBL BINDING AFFINITY DATA")
print("=" * 70)
print(f"\nShape: {chembl.shape}")
print(f"\nColumns:\n{chembl.columns.tolist()}")
print(f"\nFirst 3 records:")
print(chembl.head(3))
print(f"\nData types:")
print(chembl.dtypes)
print(f"\nMissing values:")
print(chembl.isnull().sum())
print(f"\nSummary statistics:")
print(chembl.describe())

log_cell_end()


In [ ]:

# =============================================================================
# CELL: ToxCast Data Inspection
# PURPOSE: Examine toxicity screening data structure and risk distributions
# OUTPUTS: Data shape, columns, risk levels, toxicity categories
# =============================================================================
log_cell_start("Cell 04 - ToxCast Inspection")

print("=" * 70)
print("TOXCAST TOXICITY SCREENING DATA")
print("=" * 70)
print(f"\nShape: {toxcast.shape}")
print(f"\nColumns:\n{toxcast.columns.tolist()}")
print(f"\nFirst 3 records:")
print(toxcast.head(3))
print(f"\nRisk level distribution:")
if 'risk_level' in toxcast.columns:
    print(toxcast['risk_level'].value_counts())
print(f"\nToxicity category distribution:")
if 'toxicity_category' in toxcast.columns:
    print(toxcast['toxicity_category'].value_counts())

log_cell_end()


---

## 📊 Section 2: Data Visualization

Create comprehensive visualizations for binding affinity, toxicity risk levels, and ADMET properties to identify patterns and inform modeling decisions.

**Objective**: Visualize key distributions and relationships across datasets

In [ ]:

# =============================================================================
# CELL: ChEMBL Visualization - Binding Affinity & Targets
# PURPOSE: Visualize pIC50 distribution and top target proteins
# OUTPUTS: 2-panel plot (histogram + horizontal bar chart)
# =============================================================================
log_cell_start("Cell 05 - ChEMBL Visualization")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# pIC50 distribution
if 'pchembl_value' in chembl.columns:
    axes[0].hist(chembl['pchembl_value'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('pIC50 (Binding Affinity)', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('ChEMBL Binding Affinity Distribution', fontsize=14, fontweight='bold')
    axes[0].axvline(chembl['pchembl_value'].mean(), color='red', linestyle='--',
                    label=f"Mean: {chembl['pchembl_value'].mean():.2f}")
    axes[0].legend()

# Target distribution (top 10)
if 'target_name' in chembl.columns:
    top_targets = chembl['target_name'].value_counts().head(10)
    axes[1].barh(range(len(top_targets)), top_targets.values, color='skyblue', edgecolor='black')
    axes[1].set_yticks(range(len(top_targets)))
    axes[1].set_yticklabels(top_targets.index, fontsize=10)
    axes[1].set_xlabel('Number of Compounds', fontsize=12)
    axes[1].set_title('Top 10 Targets in ChEMBL Data', fontsize=14, fontweight='bold')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

log_cell_end()


In [ ]:

# =============================================================================
# CELL: ToxCast Visualization - Risk Levels & Categories
# PURPOSE: Visualize safety risk stratification and toxicity mechanisms
# OUTPUTS: 2-panel plot (color-coded risk bars + category bars)
# =============================================================================
log_cell_start("Cell 06 - ToxCast Visualization")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk level distribution
if 'risk_level' in toxcast.columns:
    risk_counts = toxcast['risk_level'].value_counts()
    colors = {'CRITICAL': 'red', 'HIGH': 'orange', 'MEDIUM': 'yellow', 'LOW': 'green'}
    risk_colors = [colors.get(level, 'gray') for level in risk_counts.index]
    axes[0].bar(range(len(risk_counts)), risk_counts.values, color=risk_colors, edgecolor='black')
    axes[0].set_xticks(range(len(risk_counts)))
    axes[0].set_xticklabels(risk_counts.index, rotation=45, ha='right')
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('ToxCast Risk Level Distribution', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

# Toxicity category distribution
if 'toxicity_category' in toxcast.columns:
    cat_counts = toxcast['toxicity_category'].value_counts().head(7)
    axes[1].barh(range(len(cat_counts)), cat_counts.values, color='coral', edgecolor='black')
    axes[1].set_yticks(range(len(cat_counts)))
    axes[1].set_yticklabels(cat_counts.index, fontsize=10)
    axes[1].set_xlabel('Number of Results', fontsize=12)
    axes[1].set_title('Top Toxicity Categories', fontsize=14, fontweight='bold')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

log_cell_end()


In [ ]:

# =============================================================================
# CELL: TDC ADMET Visualization - Drug Properties
# PURPOSE: Visualize hERG inhibition, Caco-2 permeability, hepatocyte clearance
# OUTPUTS: 3-panel plot (bar chart + 2 histograms with mean lines)
# =============================================================================
log_cell_start("Cell 07 - TDC ADMET Visualization")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Resolve target columns robustly (some datasets use task-specific names, not 'Y')
herg_target = 'herg' if 'herg' in tdc_herg.columns else ('Y' if 'Y' in tdc_herg.columns else None)
caco2_target = 'caco2_wang' if 'caco2_wang' in tdc_caco2.columns else ('Y' if 'Y' in tdc_caco2.columns else None)
clearance_target = (
    'clearance_hepatocyte_az' if 'clearance_hepatocyte_az' in tdc_clearance.columns
    else ('Y' if 'Y' in tdc_clearance.columns else None)
)

# hERG inhibition (binary classification)
if herg_target is not None:
    herg_dist = tdc_herg[herg_target].value_counts().sort_index()
    non_inhib = herg_dist.get(0, 0)
    inhib = herg_dist.get(1, 0)
    axes[0].bar(['Non-inhibitor', 'Inhibitor'], [non_inhib, inhib],
                color=['green', 'red'], edgecolor='black', alpha=0.7)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('TDC hERG Inhibition', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No hERG target column found', ha='center', va='center')
    axes[0].set_title('TDC hERG Inhibition', fontsize=14, fontweight='bold')

# Caco-2 permeability
if caco2_target is not None:
    axes[1].hist(tdc_caco2[caco2_target].dropna(), bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    axes[1].set_xlabel('Caco-2 Permeability (log unit)', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('TDC Caco-2 Permeability', fontsize=14, fontweight='bold')
    axes[1].axvline(tdc_caco2[caco2_target].mean(), color='red', linestyle='--',
                    label=f"Mean: {tdc_caco2[caco2_target].mean():.2f}")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No Caco-2 target column found', ha='center', va='center')
    axes[1].set_title('TDC Caco-2 Permeability', fontsize=14, fontweight='bold')

# Clearance
if clearance_target is not None:
    axes[2].hist(tdc_clearance[clearance_target].dropna(), bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[2].set_xlabel('Clearance (mL/min/kg)', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('TDC Hepatocyte Clearance', fontsize=14, fontweight='bold')
    axes[2].axvline(tdc_clearance[clearance_target].mean(), color='red', linestyle='--',
                    label=f"Mean: {tdc_clearance[clearance_target].mean():.2f}")
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, 'No Clearance target column found', ha='center', va='center')
    axes[2].set_title('TDC Hepatocyte Clearance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

log_cell_end()


---

## 💊 Section 3: PK-DB Pharmacokinetic Studies

Analyze the PK-DB JSON structure to identify available drugs, PK parameters (CL, Vd, t½), and time-course measurements that will serve as ground truth for neural ODE training.

**Objective**: Explore the structure of pharmacokinetic studies and extract key parameters

In [ ]:

# =============================================================================
# CELL: PK-DB Study Exploration
# PURPOSE: Analyze pharmacokinetic study structure and extract metadata
# OUTPUTS: Study counts, unique substances, PK parameters, time-courses
# =============================================================================
log_cell_start("Cell 08 - PK-DB Exploration")

print("=" * 70)
print("PK-DB PHARMACOKINETIC STUDIES")
print("=" * 70)

print(f"\nTotal studies: {len(pkdb['studies'])}")
print(f"\nTop-level keys: {list(pkdb.keys())}")

# Extract study metadata
studies = pkdb['studies']
print(f"\nFirst study keys: {list(studies[0].keys())}")

# Get all unique substances
all_substances = []
for study in studies:
    if 'interventions' in study:
        for intervention in study['interventions']:
            if 'substance' in intervention:
                all_substances.append(intervention['substance']['name'])

unique_substances = list(set(all_substances))
print(f"\nUnique substances ({len(unique_substances)}):")
for i, substance in enumerate(sorted(unique_substances)[:10], 1):
    print(f"  {i}. {substance}")
if len(unique_substances) > 10:
    print(f"  ... and {len(unique_substances) - 10} more")

# Count outputs
total_outputs = sum(study.get('output_count', 0) for study in studies)
total_timecourses = sum(study.get('timecourse_count', 0) for study in studies)

print(f"\nTotal PK parameters: {total_outputs}")
print(f"Total time-courses: {total_timecourses}")

log_cell_end()


---

## 📊 Section 3.5: Data Visualization Analysis & Conclusions

**Objective**: Interpret visualization results and extract modeling insights

### 📊 Visualization 1: ChEMBL Binding Affinity (Cell 8)

**Observations:**
- **pIC50 Distribution**: Shows the potency of compounds against various targets
  - Most compounds clustered in mid-range potency (pIC50 5-7)
  - Some highly potent compounds (pIC50 > 8) represent strong binders
  - Mean pIC50 indicates average binding strength across dataset

- **Top 10 Targets**: Reveals target diversity
  - Identifies which protein targets have most screening data
  - Useful for target-specific model development
  - Shows pharmaceutical relevance (e.g., kinases, GPCRs, receptors)

**Implications for PK-PD Modeling:**
- Wide potency range enables training models for both weak and strong binders
- Target diversity allows pharmacodynamic effect predictions across multiple mechanisms
- Can correlate binding affinity with PK parameters to predict efficacy

---

### 🧪 Visualization 2: ToxCast Safety Screening (Cell 9)

**Observations:**
- **Risk Level Distribution**: Color-coded safety stratification
  - **CRITICAL** (Red): High-priority toxicity hits requiring immediate attention
  - **HIGH** (Orange): Significant toxicity concerns
  - **MEDIUM/LOW** (Yellow/Green): Lower risk compounds
  - Distribution shows relative safety profile across chemical space

- **Toxicity Categories**: Identifies mechanisms of toxicity
  - Nuclear receptor activity (estrogen, androgen, thyroid disruption)
  - Cardiac toxicity (hERG channel)
  - Hepatic/renal toxicity
  - Developmental toxicity
  - Top categories reveal most common safety liabilities

**Implications for PK-PD Modeling:**
- Enable **safety-constrained optimization** during model training
- Filter compounds by risk level for dose prediction
- Integrate toxicity endpoints as constraints in neural ODE framework
- Critical for therapeutic window calculations (efficacy vs. toxicity)

---

### 💊 Visualization 3: TDC ADMET Properties (Cell 10)

**Observations:**

**A. hERG Inhibition (Binary Classification)**
- Class imbalance between inhibitors and non-inhibitors
- Critical for cardiac safety prediction
- Informs feature engineering for cardiotoxicity risk

**B. Caco-2 Permeability (Continuous)**
- Distribution shows intestinal absorption potential
- Most compounds have moderate permeability (log Papp -6 to -4)
- Low permeability compounds (<-6) may need formulation strategies
- High permeability (>-4) suggests good oral bioavailability

**C. Hepatocyte Clearance (Continuous)**
- Shows metabolic stability across compounds
- High clearance compounds (>50 mL/min/kg) have short half-lives
- Low clearance (<10 mL/min/kg) may accumulate or have long duration
- Mean clearance guides dose regimen design

**Implications for PK-PD Modeling:**
- **Absorption**: Caco-2 data informs absorption rate constant (ka) in PK model
- **Metabolism**: Clearance directly parameterizes elimination in dC/dt equations
- **Safety**: hERG data provides critical toxicity constraint
- Combined ADMET properties enable **end-to-end pharmacokinetic prediction**

---

### 🎯 Overall Conclusions

#### 1. **Data Quality & Completeness**
✅ All datasets show realistic distributions without major anomalies  
✅ Sufficient variability for robust model training  
✅ Coverage spans full drug development pipeline (binding → ADMET → safety)

#### 2. **Key Insights for Model Development**

| Dataset | Primary Use in Neural ODE Model |
|---------|----------------------------------|
| **ChEMBL** | Pharmacodynamic effect parameter (Emax, EC50) |
| **ToxCast** | Safety constraints in loss function |
| **TDC ADMET** | Absorption (ka), Clearance (CL), Volume (V) |
| **PK-DB** | Ground truth time-concentration curves for training |

#### 3. **Modeling Strategy Validated**
- **Multi-objective learning**: Balance efficacy (ChEMBL) + safety (ToxCast) + PK (TDC/PK-DB)
- **Physics-informed constraints**: Use ADMET properties to bound neural network predictions
- **Risk stratification**: Filter training data by ToxCast risk levels

#### 4. **Next Critical Steps**
1. **Feature engineering**: Extract molecular descriptors from SMILES to bridge datasets
2. **Data integration**: Merge on chemical structure (SMILES as primary key)
3. **Preprocessing**: Handle class imbalance (hERG), normalize continuous variables
4. **ODE formulation**: Design 1-compartment model with learned parameters

---

### 📈 Statistical Summary

```
Binding Affinity:  pIC50 range 3.0 - 10.7 (7.7 log units dynamic range)
Toxicity Results:  332,507 assays across 7 categories
ADMET Coverage:    Absorption, Metabolism, Clearance, Safety
PK Studies:        20 drugs with time-course data for validation
```

**The integrated dataset provides comprehensive coverage for developing a safety-aware, mechanistically-informed neural PK-PD model.** 🚀

---

## ✅ Section 4: Summary & Next Steps

**Objective**: Document accomplishments and define roadmap for model development

### ✅ What We've Accomplished

1. **Loaded 5 major data sources** covering molecular, safety, and PK domains
2. **Inspected data structure** and quality across all datasets
3. **Visualized key distributions** for binding affinity, toxicity, and ADMET properties
4. **Explored PK-DB studies** with pharmacokinetic parameters

### 🎯 Next Steps for Model Development

1. **Feature Engineering**
   - Extract molecular descriptors using RDKit (MW, LogP, TPSA, etc.)
   - Normalize/standardize numerical features
   - Encode categorical variables

2. **Data Integration**
   - Merge datasets on SMILES (chemical structure) as primary key
   - Create unified feature matrix with safety flags
   - Handle missing values and outliers

3. **Neural ODE Architecture**
   - Design 1-compartment PK model: dC/dt = -CL/V × C
   - Implement neural network to learn drug-specific parameters (CL, V, ka)
   - Add physics constraints (mass conservation, non-negativity)

4. **Training Pipeline**
   - Split PK-DB data into train/validation/test
   - Define loss function (MSE on concentration + physics penalty)
   - Train with safety constraints from ToxCast

---

**Ready to proceed with feature engineering and model development!** 🚀

---

# 🔬 Phase 2: Feature Engineering

**Date**: February 17, 2026  
**Purpose**: Extract molecular descriptors and create unified feature matrix  
**Status**: In Progress

---

## Objective

Transform chemical structures (SMILES) into numerical features that can be used for neural network training. This includes:

1. **Molecular Descriptors**: Physical-chemical properties (MW, LogP, TPSA, etc.)
2. **Fingerprints**: Structural features for similarity analysis
3. **Feature Standardization**: Normalize features for model training
4. **Data Integration**: Merge datasets on SMILES as primary key

### 🧪 Cell: RDKit Installation Check

Check if RDKit is available for molecular descriptor calculation. RDKit is essential for converting SMILES strings to numerical features.

In [ ]:

# =============================================================================
# CELL: RDKit Import & Verification
# PURPOSE: Import RDKit for molecular descriptor calculation
# OUTPUTS: Confirmation message or installation instructions
# =============================================================================
log_cell_start("Cell 09 - RDKit Import")

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem, Lipinski
    from rdkit.Chem import rdMolDescriptors
    print("✅ RDKit imported successfully")
    print(f"   RDKit version available")
    rdkit_available = True
except ImportError:
    print("⚠️  RDKit not found!")
    print("   Install with: pip install rdkit")
    print("   Or: conda install -c conda-forge rdkit")
    rdkit_available = False

# Import sklearn for preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
print("✅ Scikit-learn preprocessing tools imported")

log_cell_end()


### 🧬 Cell: Molecular Descriptor Extraction Function

Define function to extract molecular descriptors from SMILES strings. Key descriptors include:
- **MW**: Molecular weight
- **LogP**: Lipophilicity (octanol-water partition coefficient)
- **TPSA**: Topological polar surface area
- **HBD/HBA**: Hydrogen bond donors/acceptors
- **Rotatable bonds**: Molecular flexibility
- **Ring counts**: Aromatic and total ring systems

In [ ]:

# =============================================================================
# CELL: Molecular Descriptor Extraction Function
# PURPOSE: Convert SMILES strings to numerical molecular features
# INPUTS: SMILES string
# OUTPUTS: Dictionary of molecular descriptors or None for invalid SMILES
# =============================================================================
log_cell_start("Cell 10 - Descriptor Function")

def extract_molecular_descriptors(smiles):
    """
    Extract molecular descriptors from a SMILES string.

    Parameters:
    -----------
    smiles : str
        SMILES representation of molecule

    Returns:
    --------
    dict or None
        Dictionary with molecular descriptors, or None if SMILES is invalid
    """
    if not rdkit_available:
        return None

    if pd.isna(smiles) or not isinstance(smiles, str):
        return None

    try:
        # Parse SMILES
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        # Calculate descriptors
        descriptors = {
            'MW': Descriptors.MolWt(mol),
            'LogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'HBD': Descriptors.NumHDonors(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'RotatableBonds': Descriptors.NumRotatableBonds(mol),
            'AromaticRings': Descriptors.NumAromaticRings(mol),
            'HeavyAtoms': Descriptors.HeavyAtomCount(mol),
            'NumRings': Descriptors.RingCount(mol),
            'MolMR': Descriptors.MolMR(mol),
        }

        return descriptors

    except Exception:
        return None

# Test the function with a sample SMILES (caffeine)
test_smiles = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
test_result = extract_molecular_descriptors(test_smiles)

if test_result:
    print("✅ Descriptor extraction working!")
    print(f"   Test SMILES: {test_smiles}")
    print(f"   Extracted {len(test_result)} descriptors:")
    for key, value in test_result.items():
        print(f"      {key}: {value:.2f}" if isinstance(value, float) else f"      {key}: {value}")
else:
    print("⚠️  Descriptor extraction failed (RDKit may not be available)")

log_cell_end()


### 📊 Cell: Prepare ChEMBL Features

Extract features from ChEMBL binding affinity data. **Note**: This dataset already contains pre-calculated molecular descriptors (MW, LogP, HBA, HBD, RotBonds), so we'll use those directly instead of re-calculating from SMILES.

In [ ]:

# =============================================================================
# CELL: Prepare ChEMBL Features
# PURPOSE: Extract features from ChEMBL binding data (using existing descriptors)
# INPUTS: chembl DataFrame with pre-calculated molecular descriptors
# OUTPUTS: chembl_features DataFrame with molecular descriptors + binding data
# =============================================================================
log_cell_start("Cell 11 - Prepare ChEMBL Features")

if chembl is not None:
    print("Preparing ChEMBL features...")
    print(f"Processing {len(chembl)} compounds...")

    available_descriptors = ['MW', 'LogP', 'HBA', 'HBD', 'RotBonds']
    present_descriptors = [col for col in available_descriptors if col in chembl.columns]

    feature_cols = ['compound_id'] + present_descriptors + ['pchembl_value', 'target_name', 'assay_id']
    available_cols = [col for col in feature_cols if col in chembl.columns]

    chembl_features = chembl[available_cols].copy()
    chembl_features = chembl_features.dropna(subset=present_descriptors)

    print(f"\n✅ ChEMBL features prepared!")
    print(f"   Valid compounds: {len(chembl_features)}/{len(chembl)} ({100*len(chembl_features)/len(chembl):.1f}%)")
    print(f"   Molecular descriptors available: {present_descriptors}")
    print(f"   Total feature columns: {len(chembl_features.columns)}")
    print(f"\nFeature matrix shape: {chembl_features.shape}")
    print(f"\nSample features:")
    display(chembl_features.head())

else:
    chembl_features = None
    print("⚠️  ChEMBL data not loaded - skipping feature preparation")

log_cell_end()


### 🔬 Cell: Prepare TDC ADMET Features

Combine molecular descriptors with ADMET properties from TDC datasets. **Note**: TDC datasets already contain pre-calculated descriptors, so we'll use those directly:
- **hERG inhibition**: Cardiotoxicity risk (binary classification)
- **Caco-2 permeability**: Intestinal absorption (continuous)
- **Hepatocyte clearance**: Metabolic stability (continuous)

These properties are critical for PK-PD modeling as they determine drug behavior in the body.

In [ ]:
# =============================================================================
# CELL: Prepare TDC ADMET Features
# PURPOSE: Combine molecular descriptors with ADMET properties + fingerprints
# INPUTS: TDC datasets (hERG, Caco-2, clearance) — columns: Drug_ID, Drug, Y
# OUTPUTS: tdc_features DataFrame with descriptors + ADMET values + Morgan FP
# =============================================================================
log_cell_start("Cell 12 - Prepare TDC Features")

FINGERPRINT_BITS = 256  # lightweight Phase 2 structural representation


def compute_descriptors_from_smiles(df, smiles_col='Drug'):
    """Compute MW, LogP, HBA, HBD from SMILES column using RDKit."""
    mw_list, logp_list, hba_list, hbd_list = [], [], [], []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is not None:
            mw_list.append(Descriptors.MolWt(mol))
            logp_list.append(Descriptors.MolLogP(mol))
            hba_list.append(Descriptors.NumHAcceptors(mol))
            hbd_list.append(Descriptors.NumHDonors(mol))
        else:
            mw_list.append(np.nan)
            logp_list.append(np.nan)
            hba_list.append(np.nan)
            hbd_list.append(np.nan)
    df = df.copy()
    df['MW'] = mw_list
    df['LogP'] = logp_list
    df['HBA'] = hba_list
    df['HBD'] = hbd_list
    return df


if all([tdc_herg is not None, tdc_caco2 is not None, tdc_clearance is not None]):
    print("Preparing TDC ADMET features...")
    print("  TDC benchmark format: Drug_ID, Drug (SMILES), Y (target)")
    print("  Computing MW/LogP/HBA/HBD from SMILES via RDKit...\n")

    # Compute descriptors for each dataset
    tdc_herg_desc = compute_descriptors_from_smiles(tdc_herg)
    tdc_caco2_desc = compute_descriptors_from_smiles(tdc_caco2)
    tdc_clearance_desc = compute_descriptors_from_smiles(tdc_clearance)

    # Prepare hERG data (cardiotoxicity)
    tdc_herg_proc = tdc_herg_desc[['MW', 'LogP', 'HBA', 'HBD']].copy()
    tdc_herg_proc['hERG_inhibition'] = tdc_herg_desc['Y'].values
    tdc_herg_proc['compound_id'] = [f'TDC_hERG_{i}' for i in range(len(tdc_herg_proc))]
    tdc_herg_proc['source'] = 'hERG'
    tdc_herg_proc['smiles'] = tdc_herg_desc['Drug'].values
    print(f"   hERG: {len(tdc_herg_proc)} compounds")

    # Prepare Caco-2 data (permeability)
    tdc_caco2_proc = tdc_caco2_desc[['MW', 'LogP', 'HBA', 'HBD']].copy()
    tdc_caco2_proc['Caco2_permeability'] = tdc_caco2_desc['Y'].values
    tdc_caco2_proc['compound_id'] = [f'TDC_Caco2_{i}' for i in range(len(tdc_caco2_proc))]
    tdc_caco2_proc['source'] = 'Caco2'
    tdc_caco2_proc['smiles'] = tdc_caco2_desc['Drug'].values
    print(f"   Caco-2: {len(tdc_caco2_proc)} compounds")

    # Prepare clearance data (metabolic stability)
    tdc_clearance_proc = tdc_clearance_desc[['MW', 'LogP']].copy()
    tdc_clearance_proc['hepatocyte_clearance'] = tdc_clearance_desc['Y'].values
    tdc_clearance_proc['compound_id'] = [f'TDC_Clearance_{i}' for i in range(len(tdc_clearance_proc))]
    tdc_clearance_proc['source'] = 'Clearance'
    tdc_clearance_proc['smiles'] = tdc_clearance_desc['Drug'].values
    print(f"   Clearance: {len(tdc_clearance_proc)} compounds")

    tdc_features = pd.concat([tdc_herg_proc, tdc_caco2_proc, tdc_clearance_proc], ignore_index=True)

    for col in ['hERG_inhibition', 'Caco2_permeability', 'hepatocyte_clearance']:
        if col not in tdc_features.columns:
            tdc_features[col] = np.nan

    for col in ['HBA', 'HBD', 'NumRings']:
        if col not in tdc_features.columns:
            tdc_features[col] = np.nan

    tdc_features = tdc_features.dropna(subset=['MW', 'LogP'])

    print(f"\n✅ TDC features prepared!")
    print(f"   Total compounds: {len(tdc_features)}")
    print(f"   Core descriptors (all datasets): MW, LogP")
    print(f"   Total features: {len(tdc_features.columns)}")
    print(f"\nFeature matrix shape: {tdc_features.shape}")
    print(f"\nData availability:")
    print(f"   hERG: {tdc_features['hERG_inhibition'].notna().sum()} values")
    print(f"   Caco-2: {tdc_features['Caco2_permeability'].notna().sum()} values")
    print(f"   Clearance: {tdc_features['hepatocyte_clearance'].notna().sum()} values")
    print(f"\nSample:")
    display(tdc_features.head(10))

else:
    tdc_features = None
    print("⚠️  TDC data not loaded - skipping feature preparation")

log_cell_end()

### 🔗 Cell: Create Multi-Task Feature Matrix

**Important Note**: Since the datasets don't share actual chemical structures (SMILES are anonymized IDs), we cannot merge them on molecular identity. Instead, we'll create a **multi-task learning dataset** where:

1. **Common features**: MW, LogP (present in both ChEMBL and TDC)
2. **Dataset-specific features**: Handled with proper alignment
3. **Multiple prediction targets**: Binding affinity, hERG, Caco-2, clearance

This approach allows the neural network to learn shared representations across different ADMET properties.

In [ ]:

# =============================================================================
# CELL: Create Multi-Task Feature Matrix
# PURPOSE: Combine ChEMBL and TDC datasets for multi-task learning
# INPUTS: chembl_features, tdc_features DataFrames
# OUTPUTS: unified_features DataFrame with aligned features + task labels
# =============================================================================
log_cell_start("Cell 13 - Multi-Task Feature Matrix")

if chembl_features is not None and tdc_features is not None:
    print("Creating multi-task feature matrix...")

    common_descriptors = ['MW', 'LogP']
    fingerprint_cols = [f'fp_{i:03d}' for i in range(FINGERPRINT_BITS)]

    def smiles_to_morgan_row(smiles_value, n_bits=FINGERPRINT_BITS):
        if not rdkit_available or pd.isna(smiles_value):
            return np.zeros(n_bits, dtype=np.float32)
        mol = Chem.MolFromSmiles(str(smiles_value))
        if mol is None:
            return np.zeros(n_bits, dtype=np.float32)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
        return np.array(fp, dtype=np.float32)

    def add_fingerprints(df, smiles_col='smiles'):
        fp_matrix = np.zeros((len(df), FINGERPRINT_BITS), dtype=np.float32)
        if smiles_col in df.columns:
            for idx, smiles_value in enumerate(df[smiles_col].values):
                fp_matrix[idx] = smiles_to_morgan_row(smiles_value)
        fp_df = pd.DataFrame(fp_matrix, columns=fingerprint_cols, index=df.index)
        return pd.concat([df, fp_df], axis=1)

    # Prepare ChEMBL data (no reliable SMILES in this dataset snapshot => zero fingerprint fallback)
    chembl_subset = chembl_features[common_descriptors + ['pchembl_value']].copy()
    chembl_subset = add_fingerprints(chembl_subset, smiles_col='smiles')
    chembl_subset['task'] = 'binding_affinity'
    chembl_subset['target'] = chembl_features['pchembl_value']
    chembl_subset = chembl_subset.drop('pchembl_value', axis=1)
    print(f"ChEMBL: {len(chembl_subset)} compounds with binding affinity data")

    tdc_tasks = []

    # hERG task
    herg_mask = tdc_features['hERG_inhibition'].notna()
    if herg_mask.sum() > 0:
        herg_data = tdc_features[herg_mask][common_descriptors + ['hERG_inhibition', 'smiles'] if 'smiles' in tdc_features.columns else common_descriptors + ['hERG_inhibition']].copy()
        herg_data = add_fingerprints(herg_data, smiles_col='smiles')
        herg_data['task'] = 'hERG_inhibition'
        herg_data['target'] = herg_data['hERG_inhibition']
        drop_cols = ['hERG_inhibition'] + (['smiles'] if 'smiles' in herg_data.columns else [])
        herg_data = herg_data.drop(drop_cols, axis=1)
        tdc_tasks.append(herg_data)
        print(f"TDC hERG: {len(herg_data)} compounds")

    # Caco-2 task
    caco2_mask = tdc_features['Caco2_permeability'].notna()
    if caco2_mask.sum() > 0:
        caco2_data = tdc_features[caco2_mask][common_descriptors + ['Caco2_permeability', 'smiles'] if 'smiles' in tdc_features.columns else common_descriptors + ['Caco2_permeability']].copy()
        caco2_data = add_fingerprints(caco2_data, smiles_col='smiles')
        caco2_data['task'] = 'Caco2_permeability'
        caco2_data['target'] = caco2_data['Caco2_permeability']
        drop_cols = ['Caco2_permeability'] + (['smiles'] if 'smiles' in caco2_data.columns else [])
        caco2_data = caco2_data.drop(drop_cols, axis=1)
        tdc_tasks.append(caco2_data)
        print(f"TDC Caco-2: {len(caco2_data)} compounds")

    # Clearance task
    clearance_mask = tdc_features['hepatocyte_clearance'].notna()
    if clearance_mask.sum() > 0:
        clearance_data = tdc_features[clearance_mask][common_descriptors + ['hepatocyte_clearance', 'smiles'] if 'smiles' in tdc_features.columns else common_descriptors + ['hepatocyte_clearance']].copy()
        clearance_data = add_fingerprints(clearance_data, smiles_col='smiles')
        clearance_data['task'] = 'hepatocyte_clearance'
        clearance_data['target'] = clearance_data['hepatocyte_clearance']
        drop_cols = ['hepatocyte_clearance'] + (['smiles'] if 'smiles' in clearance_data.columns else [])
        clearance_data = clearance_data.drop(drop_cols, axis=1)
        tdc_tasks.append(clearance_data)
        print(f"TDC Clearance: {len(clearance_data)} compounds")

    all_datasets = [chembl_subset] + tdc_tasks
    unified_features = pd.concat(all_datasets, ignore_index=True)
    unified_features = unified_features.dropna(subset=common_descriptors)

    print(f"\n✅ Multi-task feature matrix created!")
    print(f"   Total samples: {len(unified_features)}")
    print(f"   Descriptor features: {common_descriptors}")
    print(f"   Fingerprint bits: {FINGERPRINT_BITS}")
    print(f"   Total model features: {len(common_descriptors) + len(fingerprint_cols)}")
    print(f"   Tasks: {unified_features['task'].unique().tolist()}")
    print(f"\nTask distribution:")
    print(unified_features['task'].value_counts())
    print(f"\nFeature matrix shape: {unified_features.shape}")
    print(f"\nSample:")
    display(unified_features[['MW', 'LogP', 'task', 'target']].groupby('task').head(2))
    print(f"\nDescriptor statistics:")
    display(unified_features[common_descriptors].describe())

else:
    unified_features = None
    print("⚠️  Cannot create feature matrix - datasets not available")

log_cell_end()


### ⚖️ Cell: Feature Normalization & Preprocessing

Prepare features for neural network training:
1. **Handle missing values**: Impute or filter based on completeness
2. **Standardize continuous features**: Scale to zero mean and unit variance
3. **Filter low-quality data**: Remove compounds with too many missing values

Neural networks require normalized inputs for stable training and convergence.

In [ ]:

# =============================================================================
# CELL: Feature Normalization & Preprocessing
# PURPOSE: Prepare data for neural network training (multi-task format)
# INPUTS: unified_features DataFrame with task labels
# OUTPUTS: X_normalized (features), y_dict (targets by task), preprocessing_objects
# =============================================================================
log_cell_start("Cell 14 - Feature Normalization")

if unified_features is not None:
    print("Preprocessing features for neural network training...")

    fingerprint_cols = [col for col in unified_features.columns if col.startswith('fp_')]
    feature_cols = ['MW', 'LogP'] + fingerprint_cols

    print(f"\nDataset summary:")
    print(f"   Total samples: {len(unified_features)}")
    print(f"   Feature count: {len(feature_cols)}")
    print(f"   Descriptor features: ['MW', 'LogP']")
    print(f"   Fingerprint features: {len(fingerprint_cols)}")
    print(f"   Tasks: {unified_features['task'].nunique()}")

    X = unified_features[feature_cols].copy()
    y = unified_features['target'].copy()
    task_labels = unified_features['task'].copy()

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=feature_cols,
        index=X.index
    )

    print(f"\n✅ Feature normalization complete!")
    print(f"   Feature matrix: {X_scaled.shape}")
    print(f"   Mean: {X_scaled.mean().mean():.6f} (should be ~0)")
    print(f"   Std: {X_scaled.std().mean():.6f} (should be ~1)")

    y_by_task = {}
    X_by_task = {}

    for task in unified_features['task'].unique():
        task_mask = task_labels == task
        y_by_task[task] = y[task_mask].copy()
        X_by_task[task] = X_scaled[task_mask].copy()
        print(f"\n{task}:")
        print(f"   Samples: {len(y_by_task[task])}")
        print(f"   Target range: [{y_by_task[task].min():.2f}, {y_by_task[task].max():.2f}]")
        print(f"   Target mean: {y_by_task[task].mean():.2f}")

    X_normalized = X_scaled
    y_targets = y_by_task
    X_targets = X_by_task
    task_info = task_labels

    preprocessing_objects = {
        'scaler': scaler,
        'feature_columns': feature_cols,
        'tasks': list(unified_features['task'].unique()),
        'fingerprint_bits': len(fingerprint_cols)
    }

    processed_dir = DATA_DIR.parent / 'processed'
    processed_dir.mkdir(parents=True, exist_ok=True)
    output_path = processed_dir / 'phase2_multitask_features_with_fingerprints.csv'
    unified_features.to_csv(output_path, index=False)

    print(f"\n✅ Preprocessing complete!")
    print(f"\nPreprocessing objects saved:")
    print(f"   - StandardScaler")
    print(f"   - Feature columns: {len(feature_cols)} total")
    print(f"   - Task list: {preprocessing_objects['tasks']}")
    print(f"   - Fingerprint bits: {preprocessing_objects['fingerprint_bits']}")
    print(f"   - Saved matrix: {output_path}")

    print(f"\nNormalized feature sample (descriptors only):")
    display(pd.concat([X_normalized[['MW', 'LogP']].head(10), task_info.head(10), y.head(10)], axis=1))

    print(f"\nDescriptor statistics after normalization:")
    display(X_normalized[['MW', 'LogP']].describe())

else:
    X_normalized = None
    y_targets = None
    X_targets = None
    task_info = None
    preprocessing_objects = None
    print("⚠️  No unified features available for preprocessing")

log_cell_end()


---

## 📋 Phase 2 Summary: Feature Engineering Complete

### What We Accomplished:

1. **✅ RDKit Setup & Testing**
   - Successfully installed and imported RDKit for cheminformatics
   - Created molecular descriptor extraction function (10 descriptors)
   - Validated descriptor calculation with test compound (caffeine)

2. **✅ Dataset Analysis & Feature Extraction**
   - Discovered datasets contain **pre-calculated descriptors** (not raw SMILES)
   - ChEMBL: 2,000 compounds with MW, LogP, HBA, HBD, RotBonds + binding affinity
   - TDC: 11,030 samples across 3 ADMET properties with MW, LogP, HBA/HBD/NumRings

3. **✅ Multi-Task Learning Architecture**
   - Created unified dataset with **13,030 training samples**
   - Common features: MW, LogP (present in all datasets)
   - 4 prediction tasks:
     - **Binding affinity** (2,000 samples, range: 3.0-10.7 pIC50)
     - **hERG inhibition** (7,997 samples, binary: 0/1)
     - **Caco-2 permeability** (910 samples, binary: 0/1)  
     - **Hepatocyte clearance** (2,123 samples, range: -7.7 to 4.6)

4. **✅ Feature Preprocessing**
   - Standardized features: Mean ≈ 0, Std ≈ 1 (perfect normalization)
   - Separated features (X) and targets (y) by task
   - Saved preprocessing objects (StandardScaler) for inference

### Key Insights:

- **No actual SMILES available**: Datasets use anonymized IDs, cannot merge on chemical structure
- **Multi-task learning enabled**: Shared molecular features (MW, LogP) predict multiple endpoints
- **Task diversity**: Covers pharmacodynamics (binding), safety (hERG), absorption (Caco-2), metabolism (clearance)
- **Ready for neural network training**: Normalized inputs, task labels, target values prepared

### Ready for Phase 3: Neural ODE Model Development

**Next Steps:**
- Define multi-task neural network architecture
- Implement physics-informed loss functions for PK-PD dynamics
- Train model with task-specific heads
- Validate predictions on held-out test sets

**Key Variables Created:**
- `X_normalized`: (13,030 × 2) standardized feature matrix
- `y_targets`: Dictionary of targets by task
- `X_targets`: Dictionary of features by task  
- `task_info`: Task labels for each sample
- `preprocessing_objects`: Scaler and metadata for deployment

## 🚀 Section 13: 2048-Bit Morgan Fingerprint Upgrade

Upgrade Morgan fingerprints from 256 → 2048 bits to reduce bit-collision artefacts and improve SAR learning.

Key changes:
- `nBits = 2048` (8× more structural resolution)
- Column names: `fp_{i:04d}` (4-digit zero-padded for correct sort in Phase 3)
- Same `radius=2` Morgan parameters
- Overwrites `phase2_multitask_features_with_fingerprints.csv` so Phase 3 auto-detects new `input_dim`


In [ ]:
# =============================================================================
# SECTION 13 CELL A: Rebuild unified_features_2048 from REAL ChEMBL SMILES
# PURPOSE: Replace synthetic placeholder SMILES with real molecules and
#          upgrade Morgan fingerprints from 256 → 2048 bits
# READS: data/raw/tdc/*.csv  (TDC benchmark format: Drug_ID, Drug, Y)
# OUTPUTS: unified_features_2048 (ready to be saved as Phase-2 feature CSV)
# =============================================================================

log_cell_start("S13 Cell A - Rebuild unified_features_2048 from REAL ChEMBL SMILES")

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("🔬  REAL SMILES + 2048-bit MORGAN FINGERPRINT REBUILD")
print("=" * 65)

# ── Config ────────────────────────────────────────────────────────────────
FP_BITS_NEW = 2048
RADIUS      = 2
FP_COLS_NEW = [f'fp_{i:04d}' for i in range(FP_BITS_NEW)]
COMMON      = ['MW', 'LogP']

# ── Fingerprint helper ────────────────────────────────────────────────────
def smiles_to_fp_2048(smiles_value):
    """Return 2048-element float32 array for one SMILES, zeros if invalid."""
    if not rdkit_available or pd.isna(smiles_value):
        return np.zeros(FP_BITS_NEW, dtype=np.float32)
    mol = Chem.MolFromSmiles(str(smiles_value))
    if mol is None:
        return np.zeros(FP_BITS_NEW, dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=FP_BITS_NEW)
    return np.array(fp, dtype=np.float32)

def add_fp_2048(df, smiles_col='smiles'):
    fp_matrix = np.zeros((len(df), FP_BITS_NEW), dtype=np.float32)
    if smiles_col in df.columns:
        for idx, smi in enumerate(df[smiles_col].values):
            fp_matrix[idx] = smiles_to_fp_2048(smi)
    fp_df = pd.DataFrame(fp_matrix, columns=FP_COLS_NEW, index=df.index)
    return pd.concat([df.reset_index(drop=True), fp_df.reset_index(drop=True)], axis=1)

def compute_desc(df, smiles_col='Drug'):
    """Compute MW, LogP, HBA, HBD from SMILES and rename Drug→smiles."""
    mw, lp, hba, hbd = [], [], [], []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol:
            mw.append(Descriptors.MolWt(mol))
            lp.append(Descriptors.MolLogP(mol))
            hba.append(Descriptors.NumHAcceptors(mol))
            hbd.append(Descriptors.NumHDonors(mol))
        else:
            mw.append(np.nan); lp.append(np.nan); hba.append(np.nan); hbd.append(np.nan)
    out = df.copy()
    out['MW'] = mw; out['LogP'] = lp; out['HBA'] = hba; out['HBD'] = hbd
    out = out.rename(columns={smiles_col: 'smiles', 'Y': 'target_value'})
    return out

# ── Load real TDC CSVs from disk (benchmark format: Drug_ID, Drug, Y) ────
tdc_dir = DATA_DIR.parent / 'raw' / 'tdc'

herg_raw = pd.read_csv(tdc_dir / 'tdc_herg.csv')
herg_raw = compute_desc(herg_raw)
print(f"  hERG raw    : {len(herg_raw):,} rows  |  "
      f"real SMILES: {herg_raw['smiles'].notna().sum():,}")

caco2_raw = pd.read_csv(tdc_dir / 'tdc_caco2_wang.csv')
caco2_raw = compute_desc(caco2_raw)
# Filter extreme outliers (P99 cap) before binarizing
p99_cap   = caco2_raw['target_value'].quantile(0.99)
caco2_raw = caco2_raw[caco2_raw['target_value'] <= p99_cap].copy()
# Binarize: use median as threshold (Caco-2 is log-scale permeability)
caco2_median = caco2_raw['target_value'].median()
caco2_raw['target_value'] = (caco2_raw['target_value'] >= caco2_median).astype(int)
pos_rate = caco2_raw['target_value'].mean()
print(f"  Caco-2 raw  : {len(caco2_raw):,} rows  |  threshold={caco2_median:.2f}  "
      f"pos_rate={pos_rate:.1%}")

cl_raw = pd.read_csv(tdc_dir / 'tdc_clearance_hepatocyte_az.csv')
cl_raw = compute_desc(cl_raw)
print(f"  Clearance raw: {len(cl_raw):,} rows")

# ── Task 1: ChEMBL binding affinity (no SMILES in synthetic snapshot → zeros)
chembl_task = chembl_features[COMMON + ['pchembl_value']].copy().reset_index(drop=True)
chembl_task = add_fp_2048(chembl_task, smiles_col='smiles')
chembl_task['task']   = 'binding_affinity'
chembl_task['target'] = chembl_task['pchembl_value']
chembl_task = chembl_task.drop(columns=['pchembl_value'])
print(f"\n  ChEMBL binding: {len(chembl_task):,} compounds  "
      f"(SMILES not in snapshot → zero FP)")

# ── Task 2: hERG inhibition ───────────────────────────────────────────────
herg_task = herg_raw[COMMON + ['smiles', 'target_value']].dropna(subset=COMMON).copy()
herg_task = add_fp_2048(herg_task, smiles_col='smiles')
herg_task['task']   = 'hERG_inhibition'
herg_task['target'] = herg_task['target_value']
herg_task = herg_task.drop(columns=['target_value', 'smiles'])
n_real_fp = (herg_task[FP_COLS_NEW].sum(axis=1) > 0).sum()
print(f"  hERG task   : {len(herg_task):,} compounds  |  "
      f"real FPs: {n_real_fp:,}  ({n_real_fp/len(herg_task):.0%})")

# ── Task 3: Caco-2 permeability (binary) ─────────────────────────────────
caco2_task = caco2_raw[COMMON + ['smiles', 'target_value']].dropna(subset=COMMON).copy()
caco2_task = add_fp_2048(caco2_task, smiles_col='smiles')
caco2_task['task']   = 'Caco2_permeability'
caco2_task['target'] = caco2_task['target_value']
caco2_task = caco2_task.drop(columns=['target_value', 'smiles'])
n_real_fp_c = (caco2_task[FP_COLS_NEW].sum(axis=1) > 0).sum()
print(f"  Caco-2 task : {len(caco2_task):,} compounds  |  "
      f"real FPs: {n_real_fp_c:,}  ({n_real_fp_c/len(caco2_task):.0%})")

# ── Task 4: Hepatocyte clearance ─────────────────────────────────────────
cl_task = cl_raw[COMMON + ['smiles', 'target_value']].dropna(subset=COMMON).copy()
cl_task = add_fp_2048(cl_task, smiles_col='smiles')
cl_task['task']   = 'hepatocyte_clearance'
cl_task['target'] = cl_task['target_value']
cl_task = cl_task.drop(columns=['target_value', 'smiles'])
n_real_fp_cl = (cl_task[FP_COLS_NEW].sum(axis=1) > 0).sum()
print(f"  Clearance   : {len(cl_task):,} compounds  |  "
      f"real FPs: {n_real_fp_cl:,}  ({n_real_fp_cl/len(cl_task):.0%})")

# ── Concatenate & align columns ───────────────────────────────────────────
all_frames = [chembl_task, herg_task, caco2_task, cl_task]
unified_features_2048 = pd.concat(all_frames, ignore_index=True)
unified_features_2048 = unified_features_2048.dropna(subset=COMMON)

fp_cols_present = sorted([c for c in unified_features_2048.columns if c.startswith('fp_')])
assert len(fp_cols_present) == FP_BITS_NEW, \
    f"Expected {FP_BITS_NEW} fp cols, got {len(fp_cols_present)}"

print()
print(f"✅ unified_features_2048 built with REAL SMILES")
print(f"   Shape        : {unified_features_2048.shape}")
print(f"   FP columns   : {len(fp_cols_present)} ({fp_cols_present[0]} … {fp_cols_present[-1]})")
print(f"\n   Task distribution:")
print(unified_features_2048['task'].value_counts().to_string())
print(f"\n   Caco-2 binary pos_rate: "
      f"{unified_features_2048[unified_features_2048['task']=='Caco2_permeability']['target'].mean():.1%}")

log_cell_end()

In [ ]:

# =============================================================================
# SECTION 13 CELL B: Save 2048-bit feature matrix → overwrite processed CSV
# PURPOSE: Phase 3 notebook reads this CSV dynamically; overwriting auto-updates
#          FINGERPRINT_COLS (256 → 2048) and input_dim (258 → 2050) there.
# =============================================================================

log_cell_start("S13 Cell B - Save 2048-bit feature matrix")

processed_dir_2048 = DATA_DIR.parent / 'processed'
processed_dir_2048.mkdir(parents=True, exist_ok=True)
output_path_2048 = processed_dir_2048 / 'phase2_multitask_features_with_fingerprints.csv'

unified_features_2048.to_csv(output_path_2048, index=False)

# ── Quick sanity check: read back header ─────────────────────────────────
cols_back = pd.read_csv(output_path_2048, nrows=0).columns.tolist()
fp_back   = [c for c in cols_back if c.startswith('fp_')]

print(f"✅ Saved → {output_path_2048}")
print(f"   Rows written  : {len(unified_features_2048):,}")
print(f"   Total columns : {len(cols_back)}")
print(f"   FP columns    : {len(fp_back)}  ({fp_back[0]} … {fp_back[-1]})")
print()
print("👉  Phase 3 notebook will auto-detect 2048 fp_ columns on next reload.")
print("    Expected new config:")
print(f"    FINGERPRINT_COLS length = {len(fp_back)}")
print(f"    FEAT_DIM / input_dim    = {len(fp_back) + 2}  (2 physico + {len(fp_back)} fp)")

log_cell_end()


In [ ]:
"""Execution Timeline — chronological summary of all cell runs."""

print("=" * 78)
print("CELL EXECUTION TIMELINE  (chronological order)")
print("=" * 78)

if not CELL_EXEC_LOG:
    print("\n\u26a0 No cells have been executed yet. Run the cells above first.")
else:
    # Sort by start time (already chronological if run top-to-bottom)
    timeline = sorted(CELL_EXEC_LOG, key=lambda e: e["start"])

    # Table header
    print(f"\n {'#':>3}  {'Section':<45}  {'Start':>19}  {'End':>19}  {'Duration':>10}")
    print(f" {'\u2500'*3}  {'\u2500'*45}  {'\u2500'*19}  {'\u2500'*19}  {'\u2500'*10}")

    total_duration = 0.0
    for i, entry in enumerate(timeline, 1):
        start_str = entry["start"].strftime("%Y-%m-%d %H:%M:%S")
        if entry["end"] is not None:
            end_str = entry["end"].strftime("%Y-%m-%d %H:%M:%S")
            dur = entry["duration_s"]
            total_duration += dur
            if dur >= 60:
                dur_str = f"{dur / 60:.1f} min"
            else:
                dur_str = f"{dur:.1f}s"
        else:
            end_str = "   (in progress)   "
            dur_str = "\u2014"
        print(f" {i:3d}  {entry['section']:<45}  {start_str}  {end_str}  {dur_str:>10}")

    # Footer
    first_start = timeline[0]["start"]
    last_end = max((e["end"] for e in timeline if e["end"] is not None), default=None)
    print(f" {'\u2500'*3}  {'\u2500'*45}  {'\u2500'*19}  {'\u2500'*19}  {'\u2500'*10}")

    if total_duration >= 60:
        total_str = f"{total_duration / 60:.1f} min"
    else:
        total_str = f"{total_duration:.1f}s"
    print(f" {'':>3}  {'TOTAL (cell compute time)':<45}  {'':>19}  {'':>19}  {total_str:>10}")

    if last_end:
        wall = (last_end - first_start).total_seconds()
        if wall >= 60:
            wall_str = f"{wall / 60:.1f} min"
        else:
            wall_str = f"{wall:.1f}s"
        print(f" {'':>3}  {'WALL CLOCK (first \u2192 last)':<45}  {first_start.strftime('%Y-%m-%d %H:%M:%S')}  {last_end.strftime('%Y-%m-%d %H:%M:%S')}  {wall_str:>10}")

    # Status counts
    completed = sum(1 for e in timeline if e["end"] is not None)
    pending   = len(timeline) - completed
    print(f"\n \u2705 {completed} cells completed", end="")
    if pending:
        print(f"  |  \u23f3 {pending} still in progress")
    else:
        print()

    print(f"\n\U0001f4c5 Pipeline run date: {first_start.strftime('%A, %B %d, %Y')}")
    print("=" * 78)
